Crear una sesión de Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, sum
from pyspark.sql.types import IntegerType
spark = SparkSession.builder \
    .appName("Procesar archivo puertos.csv") \
    .getOrCreate()

Cargo el dataset


In [ ]:
file_path = "/content/puertos.csv"
df = spark.read.csv(file_path, header=True, inferSchema=True, sep=";")

Verifico que haya quedado bien separado


In [ ]:
print("Primeras filas del DataFrame:")
df.show()
df.printSchema()

Primeras filas del DataFrame:
+----+--------------------+-----------------+----------------+-----------------+------------+-----------------+-------------+-------------+---------+------+
| Año| Autoridad Portuaria|Graneles Líquidos|Graneles Sólidos|Mercancía General|Pesca Fresca|Avitua- llamiento|Tráfico Local|TOTAL TRAFICO|     TEUS|Buques|
+----+--------------------+-----------------+----------------+-----------------+------------+-----------------+-------------+-------------+---------+------+
|2009|            A Coruña|          6820497|       3.215.589|        1.460.292|      43.108|           93.570|      283.941|   11.916.997|    7.778|  1231|
|2009|            Alicante|           114588|       1.111.169|        1.260.064|       511.0|           24.400|            3|    2.510.735|  132.059|   935|
|2009|             Almería|             1502|       3.291.672|          542.994|       4.037|          117.334|            0|    3.957.539|    1.425|  1999|
|2009|              Avilés| 

Número de líneas que hagan referencia al puerto de Barcelona.

In [ ]:
rdd = df.rdd
barcelona_count = rdd.filter(lambda row: "Barcelona" in row["Autoridad Portuaria"]).count()
print(f"Número de líneas que hacen referencia a Barcelona: {barcelona_count}")

Número de líneas que hacen referencia a Barcelona: 48


Por cada año, buques que han entrado ordenados de mayor a menor.


In [ ]:
df.printSchema()


root
 |-- Año: integer (nullable = true)
 |-- Autoridad Portuaria: string (nullable = true)
 |-- Graneles Líquidos: integer (nullable = true)
 |-- Graneles Sólidos: string (nullable = true)
 |-- Mercancía General: string (nullable = true)
 |-- Pesca Fresca: double (nullable = true)
 |-- Avitua- llamiento: string (nullable = true)
 |-- Tráfico Local: string (nullable = true)
 |-- TOTAL TRAFICO: string (nullable = true)
 |-- TEUS: string (nullable = true)
 |-- Buques: integer (nullable = true)



In [ ]:
df.createOrReplaceTempView("puertos")
query_buques_por_anio = """
SELECT `Año`, SUM(`Buques`) as Total_Buques
FROM puertos
GROUP BY `Año`
ORDER BY Total_Buques DESC
"""
buques_por_anio = spark.sql(query_buques_por_anio)
print("Buques por año ordenados de mayor a menor:")
buques_por_anio.show(truncate=False, n=buques_por_anio.count())



Buques por año ordenados de mayor a menor:
+----+------------+
|Año |Total_Buques|
+----+------------+
|2007|130211      |
|2008|121713      |
|1999|121206      |
|2006|119819      |
|2000|119790      |
|2003|118033      |
|2005|117149      |
|2004|115983      |
|1972|114281      |
|2002|113824      |
|2009|113717      |
|1973|113686      |
|1998|112715      |
|1971|111341      |
|1976|111323      |
|1977|110736      |
|1974|110152      |
|2001|109786      |
|1975|109779      |
|1997|109341      |
|1994|107595      |
|1980|107482      |
|1978|106567      |
|1996|105153      |
|1970|104469      |
|1969|104126      |
|1968|103673      |
|1981|103457      |
|1979|103388      |
|1967|103290      |
|1962|103158      |
|1991|100287      |
|1982|99264       |
|1990|98530       |
|1965|98146       |
|1964|95318       |
|1966|95269       |
|1995|95005       |
|1992|94701       |
|1993|94538       |
|1963|92837       |
|1989|92309       |
|1983|90679       |
|1984|84652       |
|1988|83720      

Para Barcelona, media de la ‘Pesca Fresc’ para todos los años.


In [ ]:
query_pesca_fresc_barcelona = """
SELECT AVG(`Pesca Fresca`) as Media_Pesca_Fresc
FROM puertos
WHERE `Autoridad Portuaria` = 'Barcelona'
"""
media_pesca_fresc = spark.sql(query_pesca_fresc_barcelona)
print("Media de 'Pesca Fresca' para Barcelona:")
media_pesca_fresc.show()

Media de 'Pesca Fresca' para Barcelona:
+-----------------+
|Media_Pesca_Fresc|
+-----------------+
|7.515562499999999|
+-----------------+



Por cada año, total de graneles líquidos que se han manejado en los puertos
españoles.

In [ ]:
query_graneles_liquidos = """
SELECT `Año`, SUM(`Graneles Líquidos`) as Total_Graneles_Liquidos
FROM puertos
GROUP BY `Año`
ORDER BY `Año` ASC
"""
graneles_liquidos = spark.sql(query_graneles_liquidos)
print("Total de graneles líquidos por año:")
graneles_liquidos.show(truncate=False, n=graneles_liquidos.count())



Total de graneles líquidos por año:
+----+-----------------------+
|Año |Total_Graneles_Liquidos|
+----+-----------------------+
|1962|24559243               |
|1963|28832782               |
|1964|32166745               |
|1965|36449764               |
|1966|41492319               |
|1967|52633356               |
|1968|64541698               |
|1969|68506351               |
|1970|75340443               |
|1971|80989266               |
|1972|85614778               |
|1973|93587674               |
|1974|99811062               |
|1975|90440580               |
|1976|101475048              |
|1977|100031264              |
|1978|101309955              |
|1979|109498851              |
|1980|114374370              |
|1981|107393089              |
|1982|102587708              |
|1983|113064823              |
|1984|104529343              |
|1985|110499195              |
|1986|116664971              |
|1987|119104921              |
|1988|112214645              |
|1989|118153448              |
|19

In [ ]:
output_path = "/content/graneles_liquidos_completo_final.csv"

# Reparticionar el DataFrame a una sola partición
graneles_liquidos.coalesce(1).write.csv(output_path, header=True, mode="overwrite")

print(f"Archivo guardado en: {output_path}")





Archivo guardado en: /content/graneles_liquidos_completo_final.csv


In [ ]:

output_folder = "/content/graneles_liquidos_temp"
graneles_liquidos.coalesce(1).write.csv(output_folder, header=True, mode="overwrite")
print(f"Datos guardados en la carpeta temporal: {output_folder}")


In [ ]:
import pandas as pd


spark_generated_file = "/content/graneles_liquidos_temp/graneles_liquido_anio_SofiaAstigueta.csv"


df_pandas = pd.read_csv(spark_generated_file)


print("Contenido del archivo CSV cargado en pandas:")
print(df_pandas.head())
